# 🇹🇭 Thai Wikipedia RAG Chatbot

Uses `Thaiwiki_ingest.py` (scraper + Thai-aware index) and `Thaiwiki_rag_helper.py` (`RAGBase` + Gemini),
then writes the Streamlit chat UI to `Thaiapp.py`.

Keep these files in one folder: `Thaiwiki_ingest.py`, `Thaiwiki_rag_helper.py`, this notebook.
Create a `.env` file containing `GEMINI_API_KEY=your_key_here`.

Install packages from a terminal (uv venvs have no pip, so `%pip` will not work):

```bash
uv add python-dotenv streamlit google-genai minsearch pythainlp requests ipykernel
```

## 1. Check the environment

The kernel must be the project's `.venv`. Restart the kernel after installing packages.

In [ ]:
import importlib, sys

print("Python:", sys.executable)
for name in ["requests", "minsearch", "pythainlp", "google.genai", "dotenv", "streamlit"]:
    try:
        importlib.import_module(name)
        print(f"  ok       {name}")
    except ImportError as exc:
        print(f"  MISSING  {name}  ->  uv add {name.replace('.', '-').replace('dotenv', 'python-dotenv')}   ({exc})")

## 2. Load the API key

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads GEMINI_API_KEY from .env
assert os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY"), "Add GEMINI_API_KEY to your .env file"
print("API key found")

## 3. Scrape Thai Wikipedia and build the index

Each document is one passage of a page: `id`, `title`, `section`, `text`, `url`.
The first run scrapes the pages in `TITLES` (see `Thaiwiki_ingest.py`) into `thaiwiki_documents.jsonl`.
Later runs only scrape pages that are not cached yet.

In [ ]:
from collections import Counter
from Thaiwiki_ingest import build_index, load_documents

documents = load_documents()      # load_documents(refresh=True) scrapes every page again
print(f"{len(documents)} passages")
for title, n in Counter(d["title"] for d in documents).most_common():
    print(f"  {title}: {n}")

index = build_index(documents)

## 4. Create the chatbot and test it

In [ ]:
from google import genai
from Thaiwiki_rag_helper import RAGBase

client = genai.Client()
chatbot = RAGBase(index=index, llm_client=client)

print(chatbot.rag("เมืองหลวงของประเทศไทยคืออะไร"))

### Inspect what the search retrieves

If an answer is wrong, check whether the right passages were found first.

In [ ]:
query = "ประเทศไทยมีภูมิอากาศแบบไหน"
for hit in chatbot.search(query, num_results=5):
    print(f"- {hit['title']} > {hit['section']}")
    print("  ", hit["text"][:100].replace("\n", " "), "...")

### Search inside one page only

In [ ]:
for hit in chatbot.search("ก่อตั้งเมื่อไหร่", num_results=3, title="กรุงเทพมหานคร"):
    print(f"- {hit['title']} > {hit['section']}")

## 5. Write the Streamlit app

This cell writes `Thaiapp.py`. It imports the two modules above, so keep all files in the same folder.

In [ ]:
%%writefile Thaiapp.py
"""Streamlit chat UI for the Thai Wikipedia RAG chatbot.

Run:  uv run streamlit run Thaiapp.py
Needs Thaiwiki_ingest.py and Thaiwiki_rag_helper.py in the same folder,
and GEMINI_API_KEY in .env
"""
import streamlit as st
from dotenv import load_dotenv
from google import genai

from Thaiwiki_ingest import CACHE_FILE, add_page, build_index, load_documents
from Thaiwiki_rag_helper import NOT_FOUND, RAGBase

load_dotenv()  # reads GEMINI_API_KEY from .env

st.set_page_config(page_title="แชทบอทความรู้จาก Thai Wikipedia", page_icon="🇹🇭")


@st.cache_resource(show_spinner="กำลังโหลดข้อมูลและสร้างดัชนี...")
def load_chatbot():
    """Load passages (scrapes only missing pages), build the index, create the bot."""
    documents = load_documents()
    index = build_index(documents)
    client = genai.Client()  # raises if GEMINI_API_KEY is missing
    return RAGBase(index, client), documents


# ----------------------------------------------------------------- load bot
try:
    chatbot, documents = load_chatbot()
except Exception as exc:
    st.error(f"เริ่มระบบไม่สำเร็จ / Could not start the chatbot: {exc}")
    st.info(
        "ตรวจสอบว่ามี GEMINI_API_KEY ในไฟล์ .env และติดตั้ง pythainlp แล้ว\n\n"
        "Check that GEMINI_API_KEY is set in .env and that pythainlp is installed."
    )
    st.stop()

pages = sorted({d["title"] for d in documents})

# ----------------------------------------------------------------- sidebar
with st.sidebar:
    st.header("ตั้งค่า")

    if "flash" in st.session_state:
        st.success(st.session_state.pop("flash"))

    page_filter = st.selectbox("ค้นหาในหน้า", ["ทุกหน้า"] + pages)
    page_filter = None if page_filter == "ทุกหน้า" else page_filter
    num_results = st.slider("จำนวนข้อความที่ค้นหา", min_value=1, max_value=10, value=5)
    chatbot.model = st.text_input("Gemini model", value=chatbot.model)
    st.caption(f"ฐานความรู้: {len(documents)} ข้อความจาก {len(pages)} หน้า")

    with st.form("add_page", clear_on_submit=True):
        new_title = st.text_input("เพิ่มหน้า Thai Wikipedia (ชื่อหน้า)")
        submitted = st.form_submit_button("เพิ่มหน้า")
    if submitted and new_title.strip():
        try:
            with st.spinner("กำลังดึงข้อมูล..."):
                added = add_page(new_title.strip())
            st.session_state.flash = (
                f"เพิ่ม {added} ข้อความ" if added else "มีหน้านี้อยู่แล้ว"
            )
            st.cache_resource.clear()
            st.rerun()
        except Exception as exc:
            st.error(f"เพิ่มหน้าไม่สำเร็จ: {exc}")

    if st.button("ดึงข้อมูลทุกหน้าใหม่"):
        with st.spinner("กำลังดึงข้อมูลใหม่..."):
            load_documents(refresh=True)
        st.cache_resource.clear()
        st.rerun()

    if st.button("ล้างการสนทนา"):
        st.session_state.messages = []
        st.rerun()

    st.caption(f"ไฟล์แคช: {CACHE_FILE}")

# ----------------------------------------------------------------- chat
st.title("🇹🇭 แชทบอทความรู้จาก Thai Wikipedia")
st.caption("ตอบจากข้อมูล Thai Wikipedia เท่านั้น")


def show_sources(sources):
    if not sources:
        return
    with st.expander("แหล่งอ้างอิง"):
        for i, src in enumerate(sources, 1):
            st.markdown(f"{i}. [{src['title']} › {src['section']}]({src['url']})")


if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])
        if msg["role"] == "assistant":
            show_sources(msg.get("sources"))

if question := st.chat_input("ถามอะไรก็ได้ที่อยู่ใน Thai Wikipedia..."):
    st.session_state.messages.append({"role": "user", "content": question})
    with st.chat_message("user"):
        st.markdown(question)

    with st.chat_message("assistant"):
        with st.spinner("กำลังค้นหาคำตอบ..."):
            try:
                results = chatbot.search(question, num_results=num_results, title=page_filter)
                if results:
                    answer = chatbot.llm(chatbot.build_prompt(question, results))
                else:
                    answer = NOT_FOUND
            except Exception as exc:
                results, answer = [], f"เกิดข้อผิดพลาด / Error: {exc}"

        # same order as the [1], [2] numbers the model sees in its context
        sources = [
            {"title": d["title"], "section": d["section"], "url": d["url"]}
            for d in results
        ]
        if answer.strip() == NOT_FOUND:
            sources = []

        st.markdown(answer)
        show_sources(sources)

    st.session_state.messages.append(
        {"role": "assistant", "content": answer, "sources": sources}
    )

## 6. Run the chat UI

Run in a **terminal** from the same folder (running it inside the notebook blocks the kernel):

```bash
uv run streamlit run Thaiapp.py
```

Then open the forwarded port 8501.